# Colab bootstrap - sparse-view depth-prior 3DGS

Overflow host for the sweep. The 4080 is primary; this notebook runs a
**shard** of the same grid and writes into the same `runs/` tree on Drive,
so `--resume` on either host skips whatever the other already finished.

Set the shard below, then Runtime > Run all.

In [ ]:
#@title Check the GPU Colab gave us
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda)

In [ ]:
#@title Mount Drive (holds data/ and runs/ so state survives disconnects)
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/dl3dcv'  #@param {type:'string'}
os.environ['DATA_ROOT'] = f'{PROJECT}/data'
os.environ['RUNS_ROOT'] = f'{PROJECT}/runs'
os.makedirs(os.environ['RUNS_ROOT'], exist_ok=True)
print('DATA_ROOT =', os.environ['DATA_ROOT'])
print('RUNS_ROOT =', os.environ['RUNS_ROOT'])

In [ ]:
#@title Clone the repo and install
REPO = ''  #@param {type:'string'}
%cd /content
![ -d final_project ] || git clone $REPO final_project
%cd /content/final_project
!git pull --ff-only || true
!bash scripts/setup_gpu.sh

## Run a shard of the sweep

`--resume` makes this safe to re-run after a disconnect: finished runs
(those with a `metrics.json`) are skipped. Colab kills long sessions, so
expect to re-run this cell several times per stage.

In [ ]:
#@title Launch
STAGE = 'stage1'  #@param ['stage1','stage2','stage3_noise','stage3_affine']
SHARD = '1/2'     #@param {type:'string'}
!python scripts/run_sweep.py \
    --sweep configs/sweep.yaml --stage $STAGE \
    --shard $SHARD --resume --explain-pruning

In [ ]:
#@title Progress across BOTH hosts (counts what is on Drive)
import os, glob
root = os.path.join(os.environ['RUNS_ROOT'], STAGE)
done = glob.glob(os.path.join(root, '*', 'metrics.json'))
failed = glob.glob(os.path.join(root, '*', 'FAILED'))
print(f'{len(done)} finished, {len(failed)} failed under {root}')
for f in failed: print('  FAILED', os.path.basename(os.path.dirname(f)))